In [ ]:
# Cell 1: Clone or pull the private repo using a PAT entered via getpass
# (the token is never hardcoded or printed/logged), then purge any
# already-imported src.* modules so a stale cached module from an earlier
# cell run in this session can never silently run instead of the code just
# pulled, and print the commit actually checked out. Writes to
# /kaggle/working -- the only writable location on a Kaggle notebook
# instance; /kaggle/input (where the datasets live) is read-only.
import os
import subprocess
import sys
from getpass import getpass

REPO_DIR = "/kaggle/working/plantguard-v2"
pat = getpass("GitHub Personal Access Token: ")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    remote = f"https://{pat}@github.com/RUDRAIndia/plantguard-v2.git"
    subprocess.run(["git", "clone", remote, REPO_DIR], check=True)

del pat
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

purged = sorted(name for name in sys.modules if name.startswith("src"))
for name in purged:
    del sys.modules[name]
print(f"Purged {len(purged)} cached src module(s): {purged}")

commit_hash = subprocess.run(
    ["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()
print(f"Checked out commit {commit_hash}")


In [ ]:
# Cell 2: Verify GPU visibility. The whole reason for moving training to
# Kaggle is its two attached Tesla T4s -- fail loudly here rather than
# silently falling back to a CPU-only run that would go unnoticed for hours
# (CLAUDE.md rule 1). Also confirms /kaggle/input is actually mounted, which
# src/config.py's IS_KAGGLE detection depends on.
from pathlib import Path

import tensorflow as tf

if not Path("/kaggle/input").is_dir():
    raise RuntimeError(
        "/kaggle/input does not exist -- this does not look like a Kaggle "
        "notebook instance. src/config.py's IS_KAGGLE detection depends on "
        "it."
    )

gpus = tf.config.list_physical_devices("GPU")
print(f"TensorFlow {tf.__version__}")
print(f"GPUs visible: {len(gpus)}")
for gpu in gpus:
    print(f"  {gpu}")

if not gpus:
    raise RuntimeError(
        "No GPU visible to TensorFlow. Check the notebook's Settings -> "
        "Accelerator is set to a GPU option before re-running -- training "
        "on CPU here would be far too slow to be usable."
    )


In [ ]:
# Cell 3: Validate the mounted dataset inputs in place. On Kaggle,
# PlantVillage and the negatives source are read-only notebook inputs,
# already extracted -- src/config.IS_KAGGLE routes both functions below to
# validate the mount directly (38-class assertion + image-count range,
# never weakened) and record its provenance, instead of downloading
# anything. A validation failure names exactly which input is wrong and
# that it needs re-attaching -- see src/data/download.py and
# src/data/negatives.py for the Kaggle-branch detail.
from src.data import download, negatives

download.download_plantvillage()
negatives.download_negatives()


In [ ]:
# Cell 4: Ensure the dedupe/split/inventory/mapping artifacts under
# artifacts/ exist and are valid for the current split inputs. Restores
# them from the Kaggle-persisted plantguard-artifacts dataset first (the
# same dataset Cell 6 pushes checkpoints to and Cell 7 pushes these
# artifacts to) if a fresh session doesn't already have them locally, and
# only falls back to actually regenerating them (dedupe -> split ->
# inventory -> mapping, ~20 minutes) if nothing valid was restored -- the
# restored splits.json's split_input_hash is still checked against a fresh
# recomputation, so a stale restore can never be silently reused
# (src/data/prepare_artifacts.py; never weakens the check
# src/data/pipeline.py's load_splits() applies at training time).
from src.data import prepare_artifacts

prepare_artifacts.ensure_data_artifacts()

In [ ]:
# Cell 5: Run the training smoke test for one model end to end (frozen
# head phase + fine-tune phase, on a small synthetic image set), then print
# the resulting history JSON -- verifies the whole training path on Kaggle
# before committing to a real run. Same code path as
# colab/01_data_setup.ipynb's Cell 11.
import json

from src import config, train

manifest = train.run_training(model_name=config.CANDIDATE_MODELS[0], smoke=True)
print(json.dumps(manifest, indent=2))

history_path = config.ARTIFACTS_DIR / f"history_{config.CANDIDATE_MODELS[0]}.json"
print(history_path.read_text(encoding="utf-8"))


In [ ]:
# Cell 6: Real training -- every config.CANDIDATE_MODELS entry, one per fresh
# subprocess (src/train_all.py). Each model is isolated: a failure in one
# (e.g. a transient Kaggle-API error) is caught and logged, and the runner
# moves on to the next model instead of aborting the whole run the way a
# subprocess.run(..., check=True) loop would. Re-running this cell in a
# later session skips whatever's already complete (checked both locally and
# via the cross-session Kaggle-persisted state -- src/models/kaggle_persist.py)
# rather than redoing finished work. Ends with a summary table and raises if
# any model failed, so a partial run can never be mistaken for a complete one.
import subprocess
import sys

result = subprocess.run([sys.executable, "-m", "src.train_all"])
if result.returncode != 0:
    raise RuntimeError(
        f"src.train_all exited with code {result.returncode} -- at least one "
        "model failed. See the summary table printed above for which."
    )


In [ ]:
# Cell 7: Full Day-8 evaluation suite (src/evaluate/runner.py) -- model
# selection on validation macro-F1 (recomputed directly from each
# candidate's real checkpoint -- restored from the Kaggle-persisted
# plantguard-artifacts dataset if not already present locally -- never
# read from a training-log history file), PlantVillage test-set metrics
# (touched once), PlantDoc external evaluation, calibration,
# out-of-distribution rejection tuning (also updates
# android/app/src/main/assets/model_metadata.json's confidence_threshold
# ahead of the Day-9 model swap), robustness under corruption, and
# Grad-CAM sampling. Requires every config.CANDIDATE_MODELS entry to
# already have a real trained checkpoint from Cell 6 -- run this only
# after that cell has completed successfully for every candidate model.
# Writes every number in the final report to artifacts/results.json
# (CLAUDE.md rule 5); nothing downstream may hand-write a number instead
# of reading it from this file. Once results.json is written, also pushes it,
# figures/, and the dedupe/split/inventory/mapping outputs to the same
# Kaggle-persisted dataset (src/models/kaggle_persist_artifacts.py), so a
# session recycle never destroys them.
import json

from src.evaluate import runner

results = runner.run()
print(json.dumps({k: v for k, v in results.items() if k != "gradcam"}, indent=2)[:4000])


In [ ]:
# Cell 8: Day-9 export -- converts the selected model's checkpoint to a
# full-integer-quantized (uint8 in, uint8 out) LiteRT model using a
# representative dataset drawn from the TRAIN split only, verifies it
# against the SAME validation split the float model was scored on in
# Cell 7, and -- only if the macro-F1 drop is within
# config.TFLITE_MAX_MACRO_F1_DROP -- replaces the placeholder
# android/app/src/main/assets/model.tflite and model_metadata.json in
# place. If the drop exceeds tolerance, this raises: a float16
# comparison-only artifact is still written to artifacts/tflite/ and to
# results.json, but nothing is deployed to android/assets/ (the
# placeholder stays) -- that is a real failure requiring a human
# decision, never silently swallowed. Requires Cell 7 to have already
# completed. Re-pushes results.json (now with the export section) to the
# Kaggle-persisted dataset afterward, best-effort -- Cell 7's own push
# happened before this section existed.
import json

from src import config
from src.export import to_tflite
from src.models import kaggle_persist_artifacts

export_result = to_tflite.export()
print(json.dumps(export_result, indent=2))

if config.IS_KAGGLE:
    try:
        kaggle_persist_artifacts.push_data_artifacts()
    except Exception as exc:
        print(f"[cell-8] Re-push of results.json (with export section) failed: {exc}. "
              "Retry manually before the session ends.")